# Superstore Sales Analysis — Subqueries, CTEs & Window Functions

**Dataset:** [Superstore Dataset (Final)](https://www.kaggle.com/datasets/vivek468/superstore-dataset-final) — Kaggle

**Goal:** Practice SQL subqueries, CTEs, and window functions on a real retail
sales dataset to answer customer-level business questions (top/bottom
customers, single-order customers, above-average spenders, largest order per
customer).

**Engine:** SQLite (via Python's built-in `sqlite3`) — the queries are
portable to PostgreSQL/MySQL with only minor syntax changes (see notes at the
end).

**How to use this notebook**
1. Download `Sample-Superstore.csv` (or `Sample - Superstore.csv`) from the
   Kaggle link above and place it in the same folder as this notebook.
2. Run all cells top to bottom.

> Note: to build and test this pipeline without Kaggle credentials, a
> synthetically generated CSV (`Sample-Superstore.csv`) with the *same column
> schema* as the real dataset is included in this repo/environment. Swap it
> for the real Kaggle file — the code needs no changes since column names
> match.


## Step 0: Imports & Load CSV into `superstore_raw`

In [1]:
import sqlite3
import pandas as pd

CSV_PATH = "Sample-Superstore.csv"   # replace with "Sample - Superstore.csv" if using the original Kaggle filename

conn = sqlite3.connect("superstore.db")
cur = conn.cursor()

df = pd.read_csv(CSV_PATH, encoding="latin-1") if False else pd.read_csv(CSV_PATH)

# Standardize column names to snake_case for SQL friendliness
rename_map = {
    "Row ID": "row_id", "Order ID": "order_id", "Order Date": "order_date",
    "Ship Date": "ship_date", "Ship Mode": "ship_mode", "Customer ID": "customer_id",
    "Customer Name": "customer_name", "Segment": "segment", "Country": "country",
    "City": "city", "State": "state", "Postal Code": "postal_code", "Region": "region",
    "Product ID": "product_id", "Category": "category", "Sub-Category": "sub_category",
    "Product Name": "product_name", "Sales": "sales", "Quantity": "quantity",
    "Discount": "discount", "Profit": "profit"
}
df = df.rename(columns=rename_map)

df.to_sql("superstore_raw", conn, if_exists="replace", index=False)
print(f"Loaded {len(df)} rows into superstore_raw")
df.head()


Loaded 236 rows into superstore_raw


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,1,US-2023-100001,04/18/2023,04/24/2023,First Class,CU-10000,Victor Brown,Consumer,United States,Seattle,...,93130,West,PR-20032,Technology,Accessories,Premium Accessorie 354,1867.98,8,0.0,278.06
1,2,US-2023-100001,04/18/2023,04/24/2023,First Class,CU-10000,Victor Brown,Consumer,United States,Seattle,...,93130,West,PR-20053,Technology,Phones,Premium Phone 650,123.09,5,0.3,-15.76
2,3,US-2023-100001,04/18/2023,04/24/2023,First Class,CU-10000,Victor Brown,Consumer,United States,Seattle,...,93130,West,PR-20016,Office Supplies,Labels,Standard Label 231,669.70,3,0.2,23.59
3,4,US-2023-100002,01/05/2023,01/06/2023,Standard Class,CU-10000,Victor Brown,Consumer,United States,Seattle,...,14722,West,PR-20037,Technology,Phones,Premium Phone 871,1177.53,3,0.0,0.66
4,5,US-2023-100002,01/05/2023,01/06/2023,Standard Class,CU-10000,Victor Brown,Consumer,United States,Seattle,...,14722,West,PR-20059,Furniture,Chairs,Compact Chair 388,810.04,1,0.2,-50.84


## Step 1: Build `customers`, `orders`, `products` from the raw table

**Design note:** in this dataset, `City / State / Postal Code / Region`
describe the *ship-to location of an order*, not a fixed attribute of a
customer (the same `Customer ID` can ship to different cities across orders).
So those columns are modeled as order-level attributes, and `customers` keeps
only what is truly 1:1 with a customer: `customer_id`, `customer_name`,
`segment`.


In [2]:
cur.executescript('''
DROP TABLE IF EXISTS customers;
CREATE TABLE customers (
    customer_id     TEXT PRIMARY KEY,
    customer_name   TEXT,
    segment         TEXT
);

DROP TABLE IF EXISTS products;
CREATE TABLE products (
    product_id      TEXT PRIMARY KEY,
    product_name    TEXT,
    category        TEXT,
    sub_category    TEXT
);

DROP TABLE IF EXISTS orders;
CREATE TABLE orders (
    row_id          INTEGER PRIMARY KEY,
    order_id        TEXT,
    order_date      TEXT,
    ship_date       TEXT,
    ship_mode       TEXT,
    customer_id     TEXT,
    product_id      TEXT,
    country         TEXT,
    city            TEXT,
    state           TEXT,
    postal_code     TEXT,
    region          TEXT,
    sales           REAL,
    quantity        INTEGER,
    discount        REAL,
    profit          REAL
);
''')

# Populate using SELECT DISTINCT, as required by the assignment
cur.execute('''
    INSERT INTO customers (customer_id, customer_name, segment)
    SELECT DISTINCT customer_id, customer_name, segment FROM superstore_raw
''')

cur.execute('''
    INSERT INTO products (product_id, product_name, category, sub_category)
    SELECT DISTINCT product_id, product_name, category, sub_category FROM superstore_raw
''')

cur.execute('''
    INSERT INTO orders (row_id, order_id, order_date, ship_date, ship_mode,
                         customer_id, product_id, country, city, state,
                         postal_code, region, sales, quantity, discount, profit)
    SELECT DISTINCT row_id, order_id, order_date, ship_date, ship_mode,
                     customer_id, product_id, country, city, state, postal_code,
                     region, sales, quantity, discount, profit
    FROM superstore_raw
''')
conn.commit()

for t in ["customers", "products", "orders"]:
    n = cur.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t}: {n} rows")


customers: 40 rows
products: 59 rows
orders: 236 rows


## Step 2: Required Queries

### 2.1 — Orders with sales above the average sale (Subquery)


In [3]:
query = """
SELECT order_id, customer_id, product_id, sales
    FROM orders
    WHERE sales > (SELECT AVG(sales) FROM orders)
    ORDER BY sales DESC
"""
result = pd.read_sql_query(query, conn)
result


,order_id,customer_id,product_id,sales
0,US-2023-100027,CU-10011,PR-20030,6602.44
1,US-2023-100062,CU-10027,PR-20050,6516.30
2,US-2023-100057,CU-10023,PR-20057,5305.94
3,US-2023-100093,CU-10036,PR-20059,3951.51
4,US-2023-100101,CU-10039,PR-20051,3900.97
...,...,...,...,...
91,US-2023-100095,CU-10038,PR-20011,1304.80
92,US-2023-100066,CU-10029,PR-20040,1299.01
93,US-2023-100029,CU-10012,PR-20040,1283.41
94,US-2023-100074,CU-10030,PR-20042,1282.98


### 2.2 — Highest sales order for each customer (Subquery)


In [4]:
query = """
SELECT o.customer_id, o.order_id, o.sales
    FROM orders o
    WHERE o.sales = (
        SELECT MAX(o2.sales) FROM orders o2 WHERE o2.customer_id = o.customer_id
    )
    ORDER BY o.sales DESC
"""
result = pd.read_sql_query(query, conn)
result


,customer_id,order_id,sales
0,CU-10011,US-2023-100027,6602.44
1,CU-10027,US-2023-100062,6516.30
2,CU-10023,US-2023-100057,5305.94
3,CU-10036,US-2023-100093,3951.51
4,CU-10039,US-2023-100101,3900.97
5,CU-10026,US-2023-100061,3807.71
6,CU-10038,US-2023-100096,3759.38
7,CU-10010,US-2023-100025,3728.51
8,CU-10002,US-2023-100006,3602.49
9,CU-10029,US-2023-100068,3531.72


### 2.3 — Total sales per customer (CTE)


In [5]:
query = """
WITH customer_totals AS (
        SELECT customer_id, SUM(sales) AS total_sales
        FROM orders GROUP BY customer_id
    )
    SELECT c.customer_name, ROUND(ct.total_sales, 2) AS total_sales
    FROM customer_totals ct
    JOIN customers c ON c.customer_id = ct.customer_id
    ORDER BY ct.total_sales DESC
"""
result = pd.read_sql_query(query, conn)
result


,customer_name,total_sales
0,Yara Davis,22468.28
1,Emily Moore,17465.46
2,Irene Taylor,17281.51
3,Frank Moore,13153.77
4,Frank Williams,13146.15
5,Maria Lopez,13098.49
6,Irene Jones,12988.36
7,Paul Jones,11995.41
8,Xavier Wilson,11641.00
9,Darrin Davis,11634.86


### 2.4 — Customers with above-average total sales (CTE + Subquery)


In [6]:
query = """
WITH customer_totals AS (
        SELECT customer_id, SUM(sales) AS total_sales
        FROM orders GROUP BY customer_id
    )
    SELECT c.customer_name, ROUND(ct.total_sales, 2) AS total_sales
    FROM customer_totals ct
    JOIN customers c ON c.customer_id = ct.customer_id
    WHERE ct.total_sales > (SELECT AVG(total_sales) FROM customer_totals)
    ORDER BY ct.total_sales DESC
"""
result = pd.read_sql_query(query, conn)
result


,customer_name,total_sales
0,Yara Davis,22468.28
1,Emily Moore,17465.46
2,Irene Taylor,17281.51
3,Frank Moore,13153.77
4,Frank Williams,13146.15
5,Maria Lopez,13098.49
6,Irene Jones,12988.36
7,Paul Jones,11995.41
8,Xavier Wilson,11641.00
9,Darrin Davis,11634.86


### 2.5 — Rank all customers by total sales (Window Function: RANK)


In [7]:
query = """
WITH customer_totals AS (
        SELECT customer_id, SUM(sales) AS total_sales
        FROM orders GROUP BY customer_id
    )
    SELECT
        c.customer_name,
        ROUND(ct.total_sales, 2) AS total_sales,
        RANK() OVER (ORDER BY ct.total_sales DESC) AS sales_rank
    FROM customer_totals ct
    JOIN customers c ON c.customer_id = ct.customer_id
    ORDER BY sales_rank
"""
result = pd.read_sql_query(query, conn)
result


,customer_name,total_sales,sales_rank
0,Yara Davis,22468.28,1
1,Emily Moore,17465.46,2
2,Irene Taylor,17281.51,3
3,Frank Moore,13153.77,4
4,Frank Williams,13146.15,5
5,Maria Lopez,13098.49,6
6,Irene Jones,12988.36,7
7,Paul Jones,11995.41,8
8,Xavier Wilson,11641.00,9
9,Darrin Davis,11634.86,10


### 2.6 — Row number for each order within a customer (Window Function + PARTITION BY)


In [8]:
query = """
SELECT
        customer_id,
        order_id,
        sales,
        ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY sales DESC) AS order_rank_for_customer
    FROM orders
    ORDER BY customer_id, order_rank_for_customer
"""
result = pd.read_sql_query(query, conn)
result


,customer_id,order_id,sales,order_rank_for_customer
0,CU-10000,US-2023-100003,2118.97,1
1,CU-10000,US-2023-100002,2112.91,2
2,CU-10000,US-2023-100001,1867.98,3
3,CU-10000,US-2023-100002,1177.53,4
4,CU-10000,US-2023-100002,810.04,5
...,...,...,...,...
231,CU-10039,US-2023-100100,440.43,7
232,CU-10039,US-2023-100101,296.25,8
233,CU-10039,US-2023-100099,224.59,9
234,CU-10039,US-2023-100099,137.85,10


### 2.7 — Top 3 customers by total sales (Window Function)


In [9]:
query = """
WITH customer_totals AS (
        SELECT customer_id, SUM(sales) AS total_sales
        FROM orders GROUP BY customer_id
    ),
    ranked AS (
        SELECT
            c.customer_name,
            ct.total_sales,
            RANK() OVER (ORDER BY ct.total_sales DESC) AS sales_rank
        FROM customer_totals ct
        JOIN customers c ON c.customer_id = ct.customer_id
    )
    SELECT customer_name, ROUND(total_sales, 2) AS total_sales, sales_rank
    FROM ranked
    WHERE sales_rank <= 3
    ORDER BY sales_rank
"""
result = pd.read_sql_query(query, conn)
result


,customer_name,total_sales,sales_rank
0,Yara Davis,22468.28,1
1,Emily Moore,17465.46,2
2,Irene Taylor,17281.51,3


## Step 3: Final Combined Query
Customer Name + Total Sales + Rank — built with **JOIN + CTE + Window Function** together.


In [10]:
final_query = """
WITH customer_sales AS (
    SELECT o.customer_id, SUM(o.sales) AS total_sales
    FROM orders o
    GROUP BY o.customer_id
)
SELECT
    c.customer_name                              AS customer_name,
    ROUND(cs.total_sales, 2)                     AS total_sales,
    RANK() OVER (ORDER BY cs.total_sales DESC)   AS rank
FROM customer_sales cs
JOIN customers c ON c.customer_id = cs.customer_id
ORDER BY rank
"""
final_result = pd.read_sql_query(final_query, conn)
final_result


,customer_name,total_sales,rank
0,Yara Davis,22468.28,1
1,Emily Moore,17465.46,2
2,Irene Taylor,17281.51,3
3,Frank Moore,13153.77,4
4,Frank Williams,13146.15,5
5,Maria Lopez,13098.49,6
6,Irene Jones,12988.36,7
7,Paul Jones,11995.41,8
8,Xavier Wilson,11641.00,9
9,Darrin Davis,11634.86,10


## Mini Project: Customer Sales Insights

### 1. Top 5 customers


In [11]:
query = """
WITH customer_totals AS (
        SELECT customer_id, SUM(sales) AS total_sales FROM orders GROUP BY customer_id
    )
    SELECT c.customer_name, ROUND(ct.total_sales, 2) AS total_sales
    FROM customer_totals ct
    JOIN customers c ON c.customer_id = ct.customer_id
    ORDER BY ct.total_sales DESC
    LIMIT 5
"""
result = pd.read_sql_query(query, conn)
result


,customer_name,total_sales
0,Yara Davis,22468.28
1,Emily Moore,17465.46
2,Irene Taylor,17281.51
3,Frank Moore,13153.77
4,Frank Williams,13146.15


### 2. Bottom 5 customers


In [12]:
query = """
WITH customer_totals AS (
        SELECT customer_id, SUM(sales) AS total_sales FROM orders GROUP BY customer_id
    )
    SELECT c.customer_name, ROUND(ct.total_sales, 2) AS total_sales
    FROM customer_totals ct
    JOIN customers c ON c.customer_id = ct.customer_id
    ORDER BY ct.total_sales ASC
    LIMIT 5
"""
result = pd.read_sql_query(query, conn)
result


,customer_name,total_sales
0,Paul Moore,135.79
1,Aaron Williams,159.51
2,Yara Rodriguez,836.24
3,Victor Lopez,1348.40
4,Henry Williams,1619.73


### 3. Customers who made only one order


In [13]:
query = """
SELECT c.customer_name, COUNT(DISTINCT o.order_id) AS order_count
    FROM orders o
    JOIN customers c ON c.customer_id = o.customer_id
    GROUP BY c.customer_id, c.customer_name
    HAVING COUNT(DISTINCT o.order_id) = 1
"""
result = pd.read_sql_query(query, conn)
result


,customer_name,order_count
0,Yara Rodriguez,1
1,Aaron Williams,1
2,Aaron Moore,1
3,James Smith,1
4,Paul Moore,1
5,Henry Williams,1
6,Nathan Rodriguez,1
7,Victor Lopez,1
8,Wendy Rodriguez,1
9,Liam Miller,1


### 4. Customers with above-average sales


In [14]:
query = """
WITH customer_totals AS (
        SELECT customer_id, SUM(sales) AS total_sales FROM orders GROUP BY customer_id
    )
    SELECT c.customer_name, ROUND(ct.total_sales, 2) AS total_sales
    FROM customer_totals ct
    JOIN customers c ON c.customer_id = ct.customer_id
    WHERE ct.total_sales > (SELECT AVG(total_sales) FROM customer_totals)
    ORDER BY ct.total_sales DESC
"""
result = pd.read_sql_query(query, conn)
result


,customer_name,total_sales
0,Yara Davis,22468.28
1,Emily Moore,17465.46
2,Irene Taylor,17281.51
3,Frank Moore,13153.77
4,Frank Williams,13146.15
5,Maria Lopez,13098.49
6,Irene Jones,12988.36
7,Paul Jones,11995.41
8,Xavier Wilson,11641.00
9,Darrin Davis,11634.86


### 5. Highest order value per customer


In [15]:
query = """
SELECT c.customer_name, MAX(o.sales) AS highest_order_value
    FROM orders o
    JOIN customers c ON c.customer_id = o.customer_id
    GROUP BY c.customer_id, c.customer_name
    ORDER BY highest_order_value DESC
"""
result = pd.read_sql_query(query, conn)
result


,customer_name,highest_order_value
0,Xavier Wilson,6602.44
1,Wendy Rodriguez,6516.30
2,Darrin Davis,5305.94
3,Henry Jackson,3951.51
4,Paul Jones,3900.97
5,Maria Lopez,3807.71
6,Quinn Gonzalez,3759.38
7,James Smith,3728.51
8,Irene Jones,3602.49
9,Victor Garcia,3531.72


## Insights (auto-summarized below)
Run the cell below to print a short, data-driven summary. Replace the sample
CSV with the real Kaggle file and re-run for your actual submission numbers.


In [16]:
avg_sales = pd.read_sql_query("SELECT AVG(sales) AS avg_sales FROM orders", conn).iloc[0, 0]
n_customers = pd.read_sql_query("SELECT COUNT(*) AS n FROM customers", conn).iloc[0, 0]
n_orders = pd.read_sql_query("SELECT COUNT(DISTINCT order_id) AS n FROM orders", conn).iloc[0, 0]
single_order_customers = pd.read_sql_query(
    """
    SELECT COUNT(*) AS n FROM (
        SELECT customer_id FROM orders GROUP BY customer_id
        HAVING COUNT(DISTINCT order_id) = 1
    )
    """, conn).iloc[0, 0]
top_customer = final_result.iloc[0]

print(f"- Dataset contains {n_customers} unique customers and {n_orders} unique orders.")
print(f"- Average line-item sale value is ${avg_sales:,.2f}.")
print(f"- {single_order_customers} customers ({single_order_customers/n_customers:.0%}) have placed only a single order — "
      f"a retention opportunity.")
print(f"- Top customer by total sales is {top_customer['customer_name']} "
      f"(${top_customer['total_sales']:,.2f}).")


- Dataset contains 40 unique customers and 101 unique orders.
- Average line-item sale value is $1,273.71.
- 10 customers (25%) have placed only a single order — a retention opportunity.
- Top customer by total sales is Yara Davis ($22,468.28).


## Notes on porting these queries to PostgreSQL / MySQL
- `RANK()` and `ROW_NUMBER()` window function syntax is identical across
  SQLite, PostgreSQL, and MySQL 8+.
- SQLite is untyped; in Postgres/MySQL declare proper `VARCHAR` / `NUMERIC` /
  `DATE` types and parse `order_date` / `ship_date` as real dates.
- Loading the CSV: SQLite → `pandas.to_sql` (as above) or `.import`;
  PostgreSQL → `COPY superstore_raw FROM 'file.csv' CSV HEADER`;
  MySQL → `LOAD DATA INFILE`.
